In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
embeddings = np.load("../data/processed/embeddings/artist_embeddings.npy")
metadata = pd.read_csv("../data/processed/embeddings/artist_embedding_metadata.csv")

In [3]:
embeddings.shape

(6, 384)

In [4]:
metadata.head()

,artist_id,artist_name,text_for_embedding
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,"Albums: Ultramega OK. Tags: grunge, alternativ..."
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,"Albums: Bleach. Tags: grunge, alternative rock..."
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,"Albums: Adrenaline. Tags: alternative metal, n..."
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,"Albums: Ten. Tags: grunge, alternative rock, r..."
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,"Albums: Gish. Tags: alternative rock, dream po..."


In [5]:
metadata.columns

Index(['artist_id', 'artist_name', 'text_for_embedding'], dtype='str')

In [8]:
assert embeddings.shape[0] == len(metadata), "Mismatch: embeddings and metadata rows are not equal"

print("Embeddings shape:", embeddings.shape)
print("Metadata rows:", len(metadata))
print("Artists:")
print(metadata["artist_name"].tolist())

Embeddings shape: (6, 384)
Metadata rows: 6
Artists:
['Soundgarden', 'Nirvana', 'Deftones', 'Pearl Jam', 'The Smashing Pumpkins', 'Blur']


In [9]:
similarity_matrix = cosine_similarity(embeddings)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=metadata["artist_name"],
    columns=metadata["artist_name"]
)

similarity_df

artist_name,Soundgarden,Nirvana,Deftones,Pearl Jam,The Smashing Pumpkins,Blur
artist_name,,,,,,
Soundgarden,1.000000,0.711962,0.681799,0.740760,0.609996,0.684405
Nirvana,0.711962,1.000000,0.568362,0.754655,0.629098,0.675207
Deftones,0.681799,0.568362,1.000000,0.653498,0.595093,0.691868
Pearl Jam,0.740760,0.754655,0.653498,1.000000,0.663584,0.760920
The Smashing Pumpkins,0.609996,0.629098,0.595093,0.663584,1.000000,0.696780
Blur,0.684405,0.675207,0.691868,0.760920,0.696780,1.000000


In [15]:
def retrieve_similar_artists(query_artist, top_k=3):
    if query_artist not in similarity_df.index:
        raise ValueError(f"Artist '{query_artist}' not found.")

    scores = similarity_df[query_artist].sort_values(ascending=False)
    scores = scores.drop(query_artist)

    results = scores.head(top_k).reset_index()
    results.columns = ["retrieved_artist", "similarity_score"]
    results.insert(0, "query_artist", query_artist)
    results.insert(1, "rank", range(1, len(results) + 1))

    return results

In [16]:
retrieve_similar_artists("Pearl Jam", top_k=3)

,query_artist,rank,retrieved_artist,similarity_score
0,Pearl Jam,1,Blur,0.760920
1,Pearl Jam,2,Nirvana,0.754655
2,Pearl Jam,3,Soundgarden,0.740760


In [17]:
retrieve_similar_artists("Soundgarden", top_k=3)

,query_artist,rank,retrieved_artist,similarity_score
0,Soundgarden,1,Pearl Jam,0.740760
1,Soundgarden,2,Nirvana,0.711962
2,Soundgarden,3,Blur,0.684405


In [18]:
retrieve_similar_artists("Nirvana", top_k=3)

,query_artist,rank,retrieved_artist,similarity_score
0,Nirvana,1,Pearl Jam,0.754655
1,Nirvana,2,Soundgarden,0.711962
2,Nirvana,3,Blur,0.675207


In [20]:
all_results = []

for artist in metadata["artist_name"]:
    result = retrieve_similar_artists(artist, top_k=3)
    all_results.append(result)

vector_results = pd.concat(all_results, ignore_index=True)

vector_results

,query_artist,rank,retrieved_artist,similarity_score
0,Soundgarden,1,Pearl Jam,0.740760
1,Soundgarden,2,Nirvana,0.711962
2,Soundgarden,3,Blur,0.684405
3,Nirvana,1,Pearl Jam,0.754655
4,Nirvana,2,Soundgarden,0.711962
5,Nirvana,3,Blur,0.675207
6,Deftones,1,Blur,0.691868
7,Deftones,2,Soundgarden,0.681799
8,Deftones,3,Pearl Jam,0.653498
9,Pearl Jam,1,Blur,0.760920


In [21]:
vector_results.shape

(18, 4)

In [22]:
from pathlib import Path

out_path = Path("../data/processed/vector_retrieval_results.csv")

vector_results.to_csv(out_path, index=False)

out_path

WindowsPath('../data/processed/vector_retrieval_results.csv')

In [23]:
saved_results = pd.read_csv("../data/processed/vector_retrieval_results.csv")
saved_results.head()

,query_artist,rank,retrieved_artist,similarity_score
0,Soundgarden,1,Pearl Jam,0.740760
1,Soundgarden,2,Nirvana,0.711962
2,Soundgarden,3,Blur,0.684405
3,Nirvana,1,Pearl Jam,0.754655
4,Nirvana,2,Soundgarden,0.711962


#### Phase 7.1 Conclusion

This notebook converts the Phase 6 embedding similarity experiment into a reusable vector retriever baseline.

The retriever accepts a query artist and returns the top-k most similar artists based on cosine similarity between artist embedding vectors.

This phase intentionally reuses the cosine similarity logic from Phase 6, but packages it into a cleaner retrieval function with standardized output columns: query artist, rank, retrieved artist, and similarity score.

At this stage, the retriever does not use FAISS because the dataset only contains 6 seed artists. FAISS can be added later when the dataset becomes larger.

The saved vector retrieval results will be compared against graph retrieval results in the next phase.